In [ ]:
import numpy as np
import pandas as pd
from openai import OpenAI
from scipy.spatial import distance
import matplotlib.pyplot as plt
from ast import literal_eval
import re
from pydantic import BaseModel

In [ ]:
client = OpenAI(api_key=" ")

In [ ]:
def chatgpt_response(prompt):
    response = client.responses.create(
        model="gpt-4o-mini-2024-07-18",
        input = prompt
    )
    return response.output_text

In [ ]:
def get_embedding(text, model="text-embedding-ada-002"):
    edited_text = text.replace("\n", " ")   
    return client.embeddings.create(input=[edited_text], model=model).data[0].embedding

In [ ]:
model="gpt-4o-mini-2024-07-18"
main_role = """You are an experienced medical professional who has over 20 years of medical diagnosis experience.  
                   Do not return any symptoms that are not of a medical nature."""


In [ ]:
def split(prompt, text):
    class CaseSymptoms(BaseModel):
        HPI: list[str]
                
    completion = client.beta.chat.completions.parse(
                model=model,
                messages=[
                        #{"role": "system", "content": role},
                {"role": "user", "content": prompt + text},
                ],
                response_format=CaseSymptoms,
                )
        
    parsed_completion = completion.choices[0].message.parsed
    return parsed_completion.HPI

In [ ]:
datapath = " "
df = pd.read_csv(datapath)

In [ ]:
#parse through each row and extract HPIs
#need to have condition where data must not be missing
for index in df.index:
    text = df.at[index, "ProgressNoteText"]
    if pd.notna(text):
        extracted = split("Can you extract the history of present illness (HPI), which is a description of patient's signs and symptoms of their illness, section of the following patient note, verbatim: ", text)
        df.at[index, "ExtractedHPIs"] = extracted

In [ ]:
prompt2 = "Can you extract the history of present illness (HPI), which is a description of the patient's signs and symptoms of their illness, from the following emergency provider notes, verbatim: "
prompt3 = "Can you extract the 'Breif HPI and Hospital Course' section, stopping right before 'Discharge Physical Exam' for the following discharge summaries, verbatim: "
promptdate = "Can you extract the date of discharge in MM/DD/YYYY format from the following discharge summary: "

In [ ]:
#create HPI extracted columns for Discharge and a Column for Discharge Date
df["DischargeDate"] = [None] * len(df)
df["ExtractedDischargeHPIs"] = [None] * len(df)

In [ ]:
#need to extract ED visits

for index in range(0,len(df)):
    text = df.at[index, "EDProviderNoteText"]
    text2 = df.at[index, "DishcargeSummaryNoteText"]
    if pd.notna(text):
        extracted = split(prompt2, text)
        df.at[index, "ExtractedHPIs"] = extracted #saves extracted HPIs from EDNotes to Main Extracted HPI column
    if pd.notna(text2):
        extracted2 = split(prompt3, text2)
        df.at[index, "ExtractedDischargeHPIs"] = extracted2 #saves extracted DC HPIs to their own column
        extracteddate= split(promptdate, text2)
        df.at[index, "DischargeDate"] = extracteddate #saves DC date in its own column

In [ ]:
#create HPI extracted columns for Discharge and a Column for Discharge Date
df["HPIEmbeddings"] = [None] * len(df)
df["DCHPIEmbeddings"] = [None] * len(df)

In [ ]:
#extract discharge hpi and discharge dates
for index in range(0, len(df)):
    HPItext = df.at[index, "ExtractedHPIs"]
    DCtext = df.at[index, "ExtractedDischargeHPIs"]
    if pd.notna(HPItext):
        embedding = get_embedding(HPItext)
        df.at[index, "HPIEmbeddings"] = embedding #saves HPI embeddings in own column
    if pd.notna(DCtext):
        embedding2 = get_embedding(DCtext)
        df.at[index, "DCHPIEmbeddings"] = embedding2 #saves extracted DC embeddings to their own column

In [ ]:
#save dataframe
df.to_excel(datapath + " ")
df.to_csv(datapath + " ")

**Calculate Distances**

In [ ]:
df = pd.read_csv(datapath + " ")
df_ros = pd.read_csv(datapath+ " ")

In [ ]:
#need to clean up df
df_new = pd.DataFrame(columns=["PatientEncounterCSNID", "MRN", "EncounterDate", "EncounterType", "VisitType", "DepartmentName", 
                               "ChiefCompliant", "EDArrivalDateTime", "HospitalDischargeDate", "ProgressNoteAuthorSpecialty", "ExtractedHPIs", "HPIEmbeddings"])

In [ ]:
j = 0
for index in range(0, len(df)):
    embedding = df.at[index, "HPIEmbeddings"]
    dcembedding = df.at[index, "DCHPIEmbeddings"]
    print("Percent complete: " + str(index/len(df)*100))
    if pd.notna(embedding) and pd.isna(dcembedding):                   #this means only HPIEmbedding is populated
        df_new.at[j, "PatientEncounterCSNID"] = df.at[index, "PatientEncounterCSNID"]
        df_new.at[j, "MRN"] = df.at[index, "MRN"]
        df_new.at[j, "EncounterDate"] = df.at[index, "EncounterDate"]
        df_new.at[j, "EncounterType"] = df.at[index, "EncounterType"]
        df_new.at[j, "VisitType"] = df.at[index, "VisitType"]
        df_new.at[j, "DepartmentName"] = df.at[index, "DepartmentName"]
        df_new.at[j, "ChiefCompliant"] = df.at[index, "ChiefCompliantorReasonForVisit"]
        df_new.at[j, "EDArrivalDateTime"] = df.at[index, "EDArrivalDateTime"]
        df_new.at[j, "HospitalDischargeDate"] = df.at[index, "HospitalDischargeDateTime"]
        df_new.at[j, "ProgressNoteAuthorSpecialty"] = df.at[index, "ProgressNoteAuthorSpecialty"]
        df_new.at[j, "ExtractedHPIs"] = df.at[index, "ExtractedHPIs"]
        df_new.at[j, "HPIEmbeddings"] = df.at[index, "HPIEmbeddings"]
        j += 1
    elif pd.notna(embedding) and pd.notna(dcembedding):             #this means there are both ED and DC embeddings
        df_new.at[j, "PatientEncounterCSNID"] = df.at[index, "PatientEncounterCSNID"]
        df_new.at[j, "MRN"] = df.at[index, "MRN"]
        df_new.at[j, "EncounterDate"] = df.at[index, "EncounterDate"]
        df_new.at[j, "EncounterType"] = df.at[index, "EncounterType"]
        df_new.at[j, "VisitType"] = df.at[index, "VisitType"]
        df_new.at[j, "DepartmentName"] = df.at[index, "DepartmentName"]
        df_new.at[j, "ChiefCompliant"] = df.at[index, "ChiefCompliantorReasonForVisit"]
        df_new.at[j, "EDArrivalDateTime"] = df.at[index, "EDArrivalDateTime"]
        df_new.at[j, "HospitalDischargeDate"] = df.at[index, "HospitalDischargeDateTime"]
        df_new.at[j, "ProgressNoteAuthorSpecialty"] = df.at[index, "ProgressNoteAuthorSpecialty"]
        df_new.at[j, "ExtractedHPIs"] = df.at[index, "ExtractedHPIs"]
        df_new.at[j, "HPIEmbeddings"] = df.at[index, "HPIEmbeddings"]
        df_new.at[(j+1), "ExtractedHPIs"] = df.at[index, "ExtractedDischargeHPIs"]
        df_new.at[(j+1), "HPIEmbeddings"] = df.at[index, "DCHPIEmbeddings"]
        j += 2


In [ ]:
#now to calculated distances
matrix_ros = np.array(df_ros.Embeddings.apply(literal_eval).to_list())
matrix_visits = np.array(df_new.HPIEmbeddings.apply(literal_eval).to_list())

In [ ]:
matrix_ros.shape #check shape, should be length of ROS by number of encounters

In [ ]:
#creating new columns
df_new["Const"] = np.nan
df_new["HEENT"] = np.nan
df_new["Resp"] = np.nan
df_new["CV"] = np.nan
df_new["GI"] = np.nan
df_new["GU"] = np.nan
df_new["Met/Endo"] = np.nan
df_new["Neuro"] = np.nan
df_new["Psych"] = np.nan
df_new["Integ"] = np.nan
df_new["MSK"] = np.nan
df_new["Heme"] = np.nan
df_new["Immuno"] = np.nan

In [ ]:
row = 0
for i in matrix_visits:
    col = 1
    for j in matrix_ros:
        if col == 1:
            df_new.at[row, "Const"] = distance.euclidean(i,j)
        if col == 2: 
            df_new.at[row, "HEENT"] = distance.euclidean(i,j)
        if col == 3:
            df_new.at[row, "Resp"] = distance.euclidean(i,j)
        if col == 4:
            df_new.at[row, "CV"] = distance.euclidean(i,j)
        if col == 5:
            df_new.at[row, "GI"] = distance.euclidean(i,j)
        if col == 6:
            df_new.at[row, "GU"] = distance.euclidean(i,j)
        if col == 7:
            df_new.at[row, "Met/Endo"] = distance.euclidean(i,j)
        if col == 8:
            df_new.at[row, "Neuro"] = distance.euclidean(i,j)
        if col == 9:
            df_new.at[row, "Psych"] = distance.euclidean(i,j)
        if col == 10:
            df_new.at[row, "Integ"] = distance.euclidean(i,j)
        if col == 11:
            df_new.at[row, "MSK"] = distance.euclidean(i,j)
        if col == 12:
            df_new.at[row, "Heme"] = distance.euclidean(i,j)
        if col == 13:
            df_new.at[row, "Immuno"] = distance.euclidean(i,j)
        col += 1 
    row += 1

In [ ]:
df_new.to_excel(datapath + "")
df_new.to_csv(datapath + "")